In [4]:
from pathlib import Path
from datasets import load_from_disk

BASE = Path("/workspace/tiny-llm-from-scratch")

DATASET_PATH = BASE / "datasets/raw/claude_mythos"

OUTPUT_DIR = BASE / "datasets/processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
dataset = load_from_disk(str(DATASET_PATH))

dataset

DatasetDict({
    train: Dataset({
        features: ['messages', 'category', 'id', 'source', 'timestamp'],
        num_rows: 25000
    })
})

In [6]:
print(dataset)
print(dataset.keys())

DatasetDict({
    train: Dataset({
        features: ['messages', 'category', 'id', 'source', 'timestamp'],
        num_rows: 25000
    })
})
dict_keys(['train'])


In [7]:
train = dataset["train"]

print(train)

Dataset({
    features: ['messages', 'category', 'id', 'source', 'timestamp'],
    num_rows: 25000
})


In [8]:
train.features

{'messages': List({'role': Value('string'), 'content': Value('string')}),
 'category': Value('string'),
 'id': Value('string'),
 'source': Value('string'),
 'timestamp': Value('string')}

In [9]:
train[0]

{'messages': [{'role': 'user',
   'content': 'Solve or provide a rigorous proof sketch for: compute the closed-form solution for the 3-body problem approximation. Include computational verification where applicable and discuss implications for AI training dynamics.'},
  {'role': 'assistant',
   'content': 'Drawing from the autonomous, frontier-level reasoning characteristic of Claude Mythos (distilled for accessibility and precision), I approach this with multi-layered analysis, rigorous decomposition, security-first mindset, performance considerations, and ethical guardrails. \n\n**Rigorous Autonomous Reasoning Trace**\n\n**Step 1: Problem Formalization**\nThe query reduces to a complexity-theoretic separation.\n\n**Step 2: Mythos-Style Multi-Technique Attack**\n- Analytic: Closed-form derivation using generating functions / saddle-point approximation\n- Computational: Verified up to 10^12 scale with optimized parallel search (Rust + Rayon)\n- Probabilistic: PAC-style generalization b

In [10]:
for i in range(5):
    print("="*80)
    print(train[i])

{'messages': [{'role': 'user', 'content': 'Solve or provide a rigorous proof sketch for: compute the closed-form solution for the 3-body problem approximation. Include computational verification where applicable and discuss implications for AI training dynamics.'}, {'role': 'assistant', 'content': 'Drawing from the autonomous, frontier-level reasoning characteristic of Claude Mythos (distilled for accessibility and precision), I approach this with multi-layered analysis, rigorous decomposition, security-first mindset, performance considerations, and ethical guardrails. \n\n**Rigorous Autonomous Reasoning Trace**\n\n**Step 1: Problem Formalization**\nThe query reduces to a complexity-theoretic separation.\n\n**Step 2: Mythos-Style Multi-Technique Attack**\n- Analytic: Closed-form derivation using generating functions / saddle-point approximation\n- Computational: Verified up to 10^12 scale with optimized parallel search (Rust + Rayon)\n- Probabilistic: PAC-style generalization bounds wi

In [11]:
print("Total Samples:", len(train))

Total Samples: 25000


In [12]:
import re

def normalize(text):
    """
    Normalize whitespace while preserving paragraphs.
    """

    text = text.replace("\r", "")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [13]:
sample = train[0]

conversation = []

for msg in sample["messages"]:

    role = msg["role"].upper()

    content = normalize(msg["content"])

    conversation.append(f"{role}: {content}")

conversation_text = "\n\n".join(conversation)

print(conversation_text)

USER: Solve or provide a rigorous proof sketch for: compute the closed-form solution for the 3-body problem approximation. Include computational verification where applicable and discuss implications for AI training dynamics.

ASSISTANT: Drawing from the autonomous, frontier-level reasoning characteristic of Claude Mythos (distilled for accessibility and precision), I approach this with multi-layered analysis, rigorous decomposition, security-first mindset, performance considerations, and ethical guardrails. 

**Rigorous Autonomous Reasoning Trace**

**Step 1: Problem Formalization**
The query reduces to a complexity-theoretic separation.

**Step 2: Mythos-Style Multi-Technique Attack**
- Analytic: Closed-form derivation using generating functions / saddle-point approximation
- Computational: Verified up to 10^12 scale with optimized parallel search (Rust + Rayon)
- Probabilistic: PAC-style generalization bounds with data-dependent priors

**Key Result**
The derived PAC-Bayes bound tig

In [20]:
clean_conversations = []

for sample in train:

    conversation = []

    for msg in sample["messages"]:

        role = msg["role"].lower()

        content = normalize(msg["content"])

        if not content:
            continue

        if role == "user":
            conversation.append("<|user|>")
            conversation.append(content)

        elif role == "assistant":
            conversation.append("<|assistant|>")
            conversation.append(content)

    if len(conversation) >= 2:

        conversation.append("<|end|>")

        clean_conversations.append(
            "\n".join(conversation)
        )

In [21]:
print("="*60)

print("Original Samples :", len(train))

print("Clean Samples    :", len(clean_conversations))

print("="*60)

Original Samples : 25000
Clean Samples    : 25000


In [22]:
print(clean_conversations[0])

<|user|>
Solve or provide a rigorous proof sketch for: compute the closed-form solution for the 3-body problem approximation. Include computational verification where applicable and discuss implications for AI training dynamics.
<|assistant|>
Drawing from the autonomous, frontier-level reasoning characteristic of Claude Mythos (distilled for accessibility and precision), I approach this with multi-layered analysis, rigorous decomposition, security-first mindset, performance considerations, and ethical guardrails. 

**Rigorous Autonomous Reasoning Trace**

**Step 1: Problem Formalization**
The query reduces to a complexity-theoretic separation.

**Step 2: Mythos-Style Multi-Technique Attack**
- Analytic: Closed-form derivation using generating functions / saddle-point approximation
- Computational: Verified up to 10^12 scale with optimized parallel search (Rust + Rayon)
- Probabilistic: PAC-style generalization bounds with data-dependent priors

**Key Result**
The derived PAC-Bayes boun

In [23]:
OUTPUT_FILE = BASE / "datasets/processed/clean_corpus.txt"

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    for conv in clean_conversations:

        f.write(conv)

        f.write("\n")

        f.write("="*80)

        f.write("\n\n")

print("Saved")

print(OUTPUT_FILE)

Saved
/workspace/tiny-llm-from-scratch/datasets/processed/clean_corpus.txt


In [24]:
text = OUTPUT_FILE.read_text(encoding="utf-8")

print("="*60)

print("Characters :", len(text))

print("Words      :", len(text.split()))

print("Lines      :", len(text.splitlines()))

print("Size (MB)  :", round(len(text)/1024/1024,2))

print("="*60)

Characters : 51148750
Words      : 6196004
Lines      : 813542
Size (MB)  : 48.78


In [25]:
report = f"""
TinyGPT Dataset Report
======================

Dataset : Claude Mythos Distilled 25K

Original Samples : {len(train)}

Clean Samples : {len(clean_conversations)}

Characters : {len(text)}

Words : {len(text.split())}

Lines : {len(text.splitlines())}

"""

REPORT = BASE / "reports/dataset_report.txt"

REPORT.parent.mkdir(exist_ok=True)

REPORT.write_text(report, encoding="utf-8")

print(report)


TinyGPT Dataset Report

Dataset : Claude Mythos Distilled 25K

Original Samples : 25000

Clean Samples : 25000

Characters : 51148750

Words : 6196004

Lines : 813542




In [26]:
from collections import Counter

special_tokens = Counter()

for conv in clean_conversations:

    special_tokens["<|user|>"] += conv.count("<|user|>")
    special_tokens["<|assistant|>"] += conv.count("<|assistant|>")
    special_tokens["<|end|>"] += conv.count("<|end|>")

print("=" * 60)
print("Conversation Token Statistics")
print("=" * 60)

for token, count in special_tokens.items():
    print(f"{token:15} {count:,}")

print("=" * 60)

Conversation Token Statistics
<|user|>        25,000
<|assistant|>   25,000
<|end|>         25,000
